# Squirrel v2 — RF-DETR, YOLOv5, and YOLO26 comparison

Run after completing both YOLO notebooks, including their evaluation cells. The main table compares RF-DETR Medium, original YOLOv5 Medium, and YOLO26 Medium's accuracy-oriented head. YOLO26's NMS-free head is a secondary variant from the same training run.

Final accuracy uses a common COCO evaluator and frozen validation thresholds. Latency is freshly measured using a shared local CPU/GPU protocol. RF-DETR's existing reports remain unchanged. All comparison artifacts live under `output_model_comparison/runs/<comparison-id>/`.

This compares trained model recipes: YOLO uses 640 pixels, standard sampling, and up to 200 epochs; RF-DETR used 576 pixels, class-aware sampling, and up to 50 epochs. Different pretraining, capacity, losses, and postprocessing also affect results. One seed does not establish a universal architecture winner.

In [ ]:
from pathlib import Path
from IPython.display import display, HTML, IFrame
from csci_635_ml_project.baseline_comparison import resolve_runs, build_comparison

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RFDETR_RUN = ROOT / "output/runs/20260926T153802Z-v2-f35e9e"
YOLO26_RUN = None  # Explicit run path, or newest completed YOLO26 run.
YOLOV5_RUN = None  # Explicit run path, or newest completed original YOLOv5 run.
REFRESH_BENCHMARKS = True  # Fresh common CPU/GPU measurements for all models.
try:
    runs = resolve_runs(ROOT, RFDETR_RUN, YOLO26_RUN, YOLOV5_RUN)
    display({model: str(path) for model, path in runs.items()})
except FileNotFoundError as exc:
    runs = None
    print(exc)

## Verify compatibility and build the comparison

Refuse incompatible dataset fingerprints, split counts, class order, checkpoint hashes, frozen thresholds, or latency protocols. Reproduce RF-DETR's saved COCO scores before adding per-class and confusion reports. Re-measure RF-DETR latency rather than mixing its older timings with new YOLO timings.

The benchmark cells can take several minutes, especially on CPU. Accuracy and latency leaders are reported separately. Native loss curves use separate panels; their magnitudes cannot be ranked across models.

In [ ]:
if runs is None:
    print("Finish both training/evaluation notebooks, then rerun the configuration cell.")
else:
    COMPARISON, table = build_comparison(ROOT, runs, refresh_benchmarks=REFRESH_BENCHMARKS)
    display(table)
    print("Reports:", COMPARISON)
    display(IFrame(src=str((COMPARISON / "report.html").relative_to(ROOT)), width="100%", height=700))

## Saved evidence

`input_runs.json` records the exact experiments. `model_comparison.csv/json` contains validation/test accuracy, per-class aggregate metrics, epochs, duration, checkpoint size, parameters, and CPU/GPU latency. `per_class_comparison.csv`, benchmark records, accuracy/latency plots, species AP plots, and separate loss panels support inspection. Expanded RF-DETR reports are copied under this comparison folder. `findings.json` records leaders and interpretation limits; `reports.zip` packages the report.